In [17]:
using Combinatorics, ClusteredLowRankSolver, Infinity, LinearAlgebra, Nemo
include("fun.jl")

# setprecision(10^3)

LDL (generic function with 1 method)

In [18]:
# [7,1,4], [13,1,6], [19,1,8]

n,K,d = [19,1,8]

R, x = polynomial_ring(QQ, :x)
N, z = number_field(x^2 - 3, :z)

name_file=string("cerf/selfdual/$n","_$K","_$d.jl")

include(name_file) # includes Y-> set of positive matrices

print("done")

done

In [19]:
f4, size = f4_fun(n)
f2, size2 = f2_fun(n)

y_field=yijtp_fun_field(Y, n, f2, z)

w_field=(-2*y_field[f4[n, 0, 0, 0]+1]-y_field[f4[n, n, n, n]+1])/gamma_field(n,0,0,0,n,z)

# Obj. funct

obj_fun = (2^n-1)*w_field-y_field[f4[0, 0, 0, 0]+1]

println("Objective function = $obj_fun")
println("\n")

# Constraint 1

ref=0

for i in d:n-1
    max=(gamma_field(i,i,i,i,n,z)*w_field+(2*y_field[f4[i, 0, 0, 0]+1]+y_field[f4[i, i, i, i]+1]))

    ref += max
end

println("Sum of all constraints 1 = $ref")
println("\n")

# Constraint 2

ref=0 

for (i, j, t, p) in keys(f4)
    if ((t - p) % 2 == 0 && i !=0 && j != 0 && i+j-t-p >= d)

        max=y_field[f4[i,j,t,p]+1]
        
        ref += max
    end
end

println("Sum of all constraints 2 = $ref")

# All constraints should be 0 and the objective function 1

Objective function = 1


Sum of all constraints 1 = 0


Sum of all constraints 2 = 0


In [20]:
# Maximum and minimum eigenvalue of Y

Ypsd=[]

for k in 1:size2
    dim=length(Y[k][1,:])
    
    M=Matrix(undef,dim,dim)
    
    for i in 1:dim
        for j in 1:dim
            
            if typeof(Y[k][i,j])==AbsSimpleNumFieldElem
                
                M[i,j]=BigFloat(coeff(Y[k][i,j],0))+BigFloat(coeff(Y[k][i,j],1))*sqrt(big(3))
            else
                M[i,j]=Y[k][i,j]
               
        
            end
        end
    end
    
    push!(Ypsd,M)
    
end

ref_min=Inf
ref_max=0
for i in 1:size2
    mat=Matrix{BigFloat}(Ypsd[i])
    
    eigen=eigvals(mat)

    min=minimum(eigen)
    max=maximum(eigen)

    if ref_min > min
        ref_min=min
    end
    if max > ref_max
        ref_max=max
    end
end

println("Minimum eigenvalue = $ref_min\n")
println("Maximum eigenvalue = $ref_max")

Minimum eigenvalue = 1013.667973453288229387610913766014860440587233112886236525942312566833081343005

Maximum eigenvalue = 7.157502409236614115423512845418456963727744792323895585795273292177520333325366e+18


In [ ]:
# Check positivity with LDL method

ref_min=Inf

upper=rationalize(1.73206)
lower=rationalize(1.73205) 

for k in 1:size2
    A=Y[k]
    D=length(A[1,:])
    d=LDL(A,D)
    d_factor=Matrix(undef,D,1)

    for j in 1:D

        if coeff(d[j],1)//coeff(d[j],0) < 0
            factor=coeff(d[j],0)+upper*coeff(d[j],1) # upper bound sqrt(3)
        else
            factor=coeff(d[j],0)+lower*coeff(d[j],1) # lower bound sqrt(3)
        end
        
        d_factor[j]=factor
    end
    
    ref=minimum(d_factor)

    if ref_min > ref
        ref_min=ref
    end
end

ref_min_float=BigFloat(ref_min)

println("Minimum value (Rational) = $ref_min")
println("Minimum value (Float) = $ref_min_float")

Minimum value (Fraction) = 3661493230671426878088195053491646930261163123350681748481421851050482871989541643352535467928919080387027338011929389467907641747989233542083714517669595045298838289058403022110364472000522419495408046104298250848085677460074779710502154604238725862717362735499392445620184620960588487629511704025421326875239030734361706839696586668075854891737//2545096906307247747071022905045690109068799670743571110274537856938614188364101151300720804389477032989097877487574673012164950553419639956022111247088292263181723945422175611576466827425137783908097617382379465395892499056398290440394268025556512882341219615983906717123096489156419213624147720211831523394258892494762906779648000000000
Minimum value (Float) = 1.43864590051464475212485583121025905642949648929586662293572584940252138583274e+09
